# Logistic Regression


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

from kaggle_games.pipelines import get_lr_pipeline
from kaggle_games.split_data import get_split_data

In [ ]:
data_path = Path.cwd().parent / "data" / "games.json"

df = pd.read_json(data_path, orient="index")
df.replace(r"^\s*$", np.nan, regex=True, inplace=True)

X_train, X_test, y_train, y_test = get_split_data(df)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

In [ ]:
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

prep_steps = get_lr_pipeline().steps

pipe_baseline_lr = Pipeline(
    steps=prep_steps
    + [("lr", LogisticRegression(max_iter=2000, random_state=42))]
)

pipe_cw_lr = Pipeline(
    steps=prep_steps
    + [
        (
            "lr",
            LogisticRegression(
                class_weight="balanced", max_iter=2000, random_state=42
            ),
        )
    ]
)

pipe_smote_lr = ImbPipeline(
    steps=prep_steps
    + [
        ("smote", SMOTE(random_state=42)),
        ("lr", LogisticRegression(max_iter=2000, random_state=42)),
    ]
)

param_grid_lr = {
    "lr__C": [0.01, 0.1, 1.0, 10.0],
    "lr__solver": ["lbfgs", "liblinear"],
}


def run_search_lr(pipeline, name):
    print(f"--- Tuning {name} ---")
    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_grid_lr,
        n_iter=5,
        cv=cv_strategy,
        scoring="roc_auc",
        random_state=42,
        n_jobs=None,
        verbose=1,
    )
    search.fit(X_train, y_train)
    return search.best_estimator_


# Execute Tuning
best_baseline_lr = run_search_lr(pipe_baseline_lr, "LR Baseline")
best_cw_lr = run_search_lr(pipe_cw_lr, "LR Class Weights")
best_smote_lr = run_search_lr(pipe_smote_lr, "LR SMOTE")

models_lr = {
    "Baseline": best_baseline_lr,
    "Class Weights": best_cw_lr,
    "SMOTE": best_smote_lr,
}


def evaluate_model(model, X_test, y_test, title):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(f"\n=== {title} ===")
    print(classification_report(y_test, y_pred))

    return {
        "y_pred": y_pred,
        "y_proba": y_proba,
        "auc": roc_auc_score(y_test, y_proba),
        "f1": f1_score(y_test, y_pred),
        "prec": precision_score(y_test, y_pred),
        "rec": recall_score(y_test, y_pred),
    }


eval_results_lr = {}
for name, model in models_lr.items():
    eval_results_lr[name] = evaluate_model(model, X_test, y_test, f"LR {name}")

plt.figure(figsize=(10, 7))
colors = {"Baseline": "gray", "Class Weights": "blue", "SMOTE": "red"}

for name, res in eval_results_lr.items():
    precision, recall, _ = precision_recall_curve(y_test, res["y_proba"])
    plt.plot(
        recall,
        precision,
        label=f"{name} (F1 = {res['f1']:.3f})",
        color=colors[name],
        linewidth=2,
    )

plt.title(
    "LR Precision-Recall Curve Comparison", fontweight="bold", fontsize=14
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (name, res) in enumerate(eval_results_lr.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, res["y_pred"], ax=axes[i], cmap="Blues", colorbar=False
    )
    axes[i].set_title(
        f"LR {name}\nF1: {res['f1']:.2f} | Recall: {res['rec']:.2f}"
    )

plt.tight_layout()
plt.show()